# [SQL 재현] 2023년 의료기관별 시군구별 진료비 분석

## 단계: 02. 탐색적 데이터 분석(EDA) — SQL 재현
- 목표: PY_02에서 pandas로 만든 파생변수·상위 지역·시도별 집계를 SQL로 다시 작성하고, 결과가 PY_02와 같은지 대조한다.
- 환경: Jupyter Notebook + sqlite3 (SQL_01에서 만든 sql_practice.db의 hira 테이블 사용)
- 대조 기준: PY_02_EDA.ipynb 실행 결과
- 범위 밖: 시각화·상관계수·왜도(Python 분석 영역), 로그 변환(SQL_03에서 확인)

### 2.1 환경 설정
#### 2.1-1 DB 연결 및 저장된 테이블 확인
- SQL_01에서 만든 sql_practice.db에 다시 연결한다. 테이블이 DB 파일에 저장돼 있으므로 CSV를 다시 읽을 필요가 없다.
- `sqlite_master`: SQLite가 DB 안의 테이블 목록을 기록해 두는 시스템 표

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'C:\data\sql_practice.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

,name
0,hira


### 2.2 파생변수 생성
#### 2.2-1 파생변수를 포함한 분석용 테이블 만들기
- PY_02 2.1 파생변수 대응 (SAS `DATA hira.analysis; SET hira.raw;` 대응)
- `CREATE TABLE 새테이블 AS SELECT ...`: 조회 결과를 새 테이블로 저장
- 정수 열끼리 나눌 때는 `CAST(열 AS REAL)`로 소수형으로 바꾼 뒤 나눈다 (SQL_01 1.4-1 결과 반영)
- 숫자로 시작하는 이름은 SQL에서 매번 따옴표가 필요하므로, SAS와 같이 `일인당_`으로 시작

In [3]:
conn.execute("DROP TABLE IF EXISTS hira_eda")
conn.execute("""
CREATE TABLE hira_eda AS
SELECT *,
      ROUND(CAST(요양급여비용총액 AS REAL) / 환자수 / 10000, 1) AS 일인당_진료비_만원,
      ROUND(CAST(입내원일수 AS REAL) / 환자수, 2) AS 일인당_내원일수,
      ROUND(요양급여비용총액 / 100000000.0, 1) AS 진료비_억원,
      ROUND(CAST(요양급여비용총액 AS REAL) / 환자수, 0) AS 일인당_진료비
FROM hira
""")
conn.commit()

#### 2.2-2 새 테이블 확인
- 행 수가 251로 유지됐는지, 파생변수에 비어 있는 값(NULL)이 없는지 확인
- 나눗셈이 계산되지 못한 행(예: 0으로 나누기)이 있으면 그 칸은 NULL이 된다

In [4]:
q = """
SELECT COUNT(*) AS 행수,
       COUNT(일인당_진료비_만원) AS 진료비_값있는행,
       COUNT(일인당_내원일수) AS 내원일수_값있는행
FROM hira_eda
"""
pd.read_sql(q, conn)

,행수,진료비_값있는행,내원일수_값있는행
0,251,251,251


### 2.3 주요 지표 상위 지역 탐색
#### 2.3-1 1인당 진료비 상위 10개 시군구
- PY_02 2.2 `sort_values('1인당_진료비_만원', ascending=False).head(10)` 대응
- `ORDER BY 열 DESC`(큰 값부터 정렬) + `LIMIT 10`

In [5]:
q = """
SELECT 시도, 시군구, 일인당_진료비_만원, 일인당_내원일수
FROM hira_eda
ORDER BY 일인당_진료비_만원 DESC
LIMIT 10
"""
pd.read_sql(q, conn)

,시도,시군구,일인당_진료비_만원,일인당_내원일수
0,전남,화순군,280.6,14.22
1,부산,부산서구,189.6,9.13
2,울산,울산동구,156.2,12.38
3,서울,서대문구,150.0,8.04
4,전북,익산시,144.0,17.09
5,대구,대구남구,140.4,8.64
6,경남,진주시,132.8,14.26
7,광주,광주동구,132.3,8.04
8,경북,안동시,131.3,13.72
9,대전,대전중구,128.0,10.10


#### 2.3-2 총 진료비 상위 5개 시군구
- PY_02 2.5 EDA 1 `nlargest(5, '요양급여비용총액(선별포함)')` 대응
- 정렬 기준(원 단위 총액)과 화면에 보여주는 열(억원)이 달라도 된다

In [6]:
q = """
SELECT 시도, 시군구, 진료비_억원, 환자수, 일인당_진료비
FROM hira_eda
ORDER BY 요양급여비용총액 DESC
LIMIT 5
"""
pd.read_sql(q, conn)

,시도,시군구,진료비_억원,환자수,일인당_진료비
0,서울,강남구,29623.5,3182688,930771.0
1,서울,송파구,25385.4,1984817,1278980.0
2,경기,성남분당구,19049.1,1710142,1113889.0
3,서울,종로구,17066.9,1434102,1190073.0
4,서울,서대문구,16779.4,1118858,1499686.0


### 2.4 시도별 요약 통계
#### 2.4-1 시도별 1인당 진료비 평균·총진료비·시군구 수
- PY_02 2.5의 EDA 2(시도별 1인당 평균 진료비, 원)와 `sido_agg`(만원·억원·시군구수)를 한 쿼리로 대응
- `GROUP BY 시도` + `AVG`, `SUM`, `COUNT`, 평균 내림차순 정렬

In [9]:
q = """
SELECT 시도,
       ROUND(AVG(일인당_진료비_만원), 1) AS 평균_1인당_진료비_만원,
       ROUND(AVG(일인당_진료비), 0) AS 평균_1인당_진료비_원,
       ROUND(SUM(진료비_억원), 1) AS 총진료비_억원,
       COUNT(*) AS 시군구수
FROM hira_eda
GROUP BY 시도
ORDER BY 평균_1인당_진료비_만원 DESC
"""
pd.read_sql(q, conn)

,시도,평균_1인당_진료비_만원,평균_1인당_진료비_원,총진료비_억원,시군구수
0,광주,92.7,926887.0,30984.6,5
1,전남,87.1,871400.0,28372.6,22
2,부산,85.7,856898.0,67035.9,16
3,울산,84.2,842205.0,17464.9,5
4,전북,83.5,835116.0,30600.0,15
5,서울,83.1,830685.0,228522.1,25
6,경남,81.2,812343.0,50356.1,22
7,제주,77.8,777204.0,8810.9,2
8,대전,76.7,767272.0,27506.3,5
9,인천,76.4,764370.0,44876.6,10


### 2.5 [SQL 응용] 전국 평균보다 1인당 진료비가 높은 시도
- PY_02에 없던 새 조회. HAVING과 서브쿼리를 함께 쓰는 연습.
- 전국 평균 = 251개 시군구 1인당 진료비(만원)의 단순평균 (총진료비 ÷ 총환자수로 구하는 가중평균과는 계산 방식이 다름)
#### 2.5-1 전국 평균 확인

In [10]:
pd.read_sql("SELECT AVG(일인당_진료비_만원) AS 전국평균 FROM hira_eda", conn)

,전국평균
0,75.032271


#### 2.5-2 전국 평균보다 높은 시도만 남기기
- `HAVING`: 시도별로 묶어 계산한 평균에 조건을 건다
- 괄호 안 서브쿼리: 전국 평균 값 하나를 돌려준다 (2.5-1과 같은 쿼리)
- 비교는 반올림 전 평균으로 한다 (반올림 후 비교하면 경계에 있는 시도가 뒤바뀔 수 있음)

In [12]:
q = """
SELECT 시도,
       ROUND(AVG(일인당_진료비_만원), 1) AS 평균_만원,
       COUNT(*) AS 시군구수
FROM hira_eda
GROUP BY 시도
HAVING AVG(일인당_진료비_만원) > (SELECT AVG(일인당_진료비_만원) FROM hira_eda)
ORDER BY 평균_만원 DESC
"""
pd.read_sql(q, conn)

,시도,평균_만원,시군구수
0,광주,92.7,5
1,전남,87.1,22
2,부산,85.7,16
3,울산,84.2,5
4,전북,83.5,15
5,서울,83.1,25
6,경남,81.2,22
7,제주,77.8,2
8,대전,76.7,5
9,인천,76.4,10
